In [17]:
#%pip install datasets -q

In [18]:
#%pip installsentence-transformers transformers

In [19]:
#%pip install arabert fastembed

In [20]:
#%pip install qdrant-client langchain langchain-community langchain-qdrant

In [21]:
from datasets import load_dataset

dataset = load_dataset("khalidalt/SANAD", "default", split="train")


In [22]:
print(dataset)

Dataset({
    features: ['article', 'category'],
    num_rows: 99810
})


In [23]:
dataset[0]

{'article': '\nالمخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف\n\nالحدث.نت\n\nالتقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يرأس الوفد الحكومي للمفاوضات اليمنية أجندة وجدول أعمال مشاورات جنيف، بالإضافة إلى ملاحظات عامة وأخرى تفصيلية على المسودة المقترحة للمشاورات.وشدد المخلافي على ضرورة  تطبيق الميليشيات الانقلابية لقرار مجلس الأمن 2216  والقرارات ذات الصلة، مجدداً على  حرص الحكومة اليمنية على السلام.وأكد وزير الخارجية اليمني جدية السلطة الشرعية في التعاطي الايجابي مع كافة الجهود الدولية الهادفة الى تحقيق السلام وتطبيق قرارات الشرعية الدولية.\xa0\n',
 'category': 'Politics'}

In [24]:
import re

In [25]:
def clean_text(text):
    text = re.sub(r'[إأآا]', 'ا', text)
    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    # Remove tatweel
    text = re.sub(r'\u0640', '', text)
    # Remove non-Arabic characters except spaces and punctuation
    text = re.sub(r'[^\u0600-\u06FF\s\.\،\؟\!]', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [26]:
sample = dataset[0]['article']
sample =sample[:200]
cleaned_sample = clean_text(sample)
after_cleaning = cleaned_sample
before_cleaning = sample

In [27]:
print("Before cleaning:")
print(before_cleaning)
print("\nAfter cleaning:")
print(after_cleaning)

Before cleaning:

المخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف

الحدث.نت

التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي

After cleaning:
المخلافي يلتقي ولد الشيخ ويقدم جدول اعمال مشاورات جنيف الحدث.نت التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الاممي اسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي


In [28]:
# ! pip install qdrant-client langchain langchain-community langchain-qdrant sentence-transformers transformers datasets arabert fastembed -q

In [29]:
# !pip install langchain-text-splitters -q

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '،', '.', ' ']
)

In [31]:
def prepare_documents(dataset,max_articles=5000):
    docs=[]
    for i, item in enumerate(dataset):
        if i >= max_articles:
            break
        text=clean_text(item['article'])
        if len(text) < 50:
            continue
        chunks = text_splitter.split_text(text)
        for j,chunk in enumerate(chunks):
            docs.append({
                'id': f"{[i]}_{j}",
                'text': chunk,
                'category':item.get('category', 'unknown'),
                'source_idx': i
            })
    return docs
docs = prepare_documents(dataset)
print(f"total chunks: {len(docs)}")
print(docs[0])

total chunks: 30148
{'id': '[0]_0', 'text': 'المخلافي يلتقي ولد الشيخ ويقدم جدول اعمال مشاورات جنيف الحدث.نت التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الاممي اسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يراس الوفد الحكومي للمفاوضات اليمنية اجندة وجدول اعمال مشاورات جنيف', 'category': 'Politics', 'source_idx': 0}


In [32]:
print(dataset.column_names)
print(dataset[0])

['article', 'category']
{'article': '\nالمخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف\n\nالحدث.نت\n\nالتقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يرأس الوفد الحكومي للمفاوضات اليمنية أجندة وجدول أعمال مشاورات جنيف، بالإضافة إلى ملاحظات عامة وأخرى تفصيلية على المسودة المقترحة للمشاورات.وشدد المخلافي على ضرورة  تطبيق الميليشيات الانقلابية لقرار مجلس الأمن 2216  والقرارات ذات الصلة، مجدداً على  حرص الحكومة اليمنية على السلام.وأكد وزير الخارجية اليمني جدية السلطة الشرعية في التعاطي الايجابي مع كافة الجهود الدولية الهادفة الى تحقيق السلام وتطبيق قرارات الشرعية الدولية.\xa0\n', 'category': 'Politics'}


In [33]:
#%pip install sentence-transformers -q

In [34]:
#%pip install langchain-community -q

In [35]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, SparseVectorParams, SparseIndexParams)
from langchain_community.embeddings import HuggingFaceEmbeddings

In [36]:
client = QdrantClient(path="./qdrant_db")

In [37]:
#! pip install sentence-transformers

In [38]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

C:\Users\saleen\AppData\Local\Temp\ipykernel_20232\4233599945.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1329.37it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationshi

In [39]:
test = embeddings.embed_query("ماذا حدث؟")
print(test)

[0.036370899528265, 0.04516447335481644, -0.019641390070319176, -0.024854108691215515, 0.010806666687130928, 0.010313254781067371, -0.04588238149881363, 0.01884564384818077, -0.059371672570705414, -0.007454446982592344, -0.009409804828464985, -0.006403196137398481, -0.03919430077075958, 0.0031174193136394024, 0.005650704260915518, -0.00980814453214407, 0.0032287943176925182, -0.031219758093357086, -0.04851827770471573, 0.007481336127966642, 0.030651237815618515, -0.001985131297260523, 0.01922815665602684, -0.05994177237153053, -0.01443719957023859, 0.0058554792776703835, 0.038992539048194885, 0.021019157022237778, -0.011374900117516518, -0.042599957436323166, -0.1011614054441452, 0.019924188032746315, 0.031495992094278336, 0.00029182550497353077, -0.0022383772302418947, -0.02202344685792923, 0.04692642763257027, 0.025013098493218422, 0.042713481932878494, -0.009916993789374828, -0.013580050319433212, 0.00875004194676876, -0.013812722638249397, -0.006103650201112032, -0.0007785083143971

In [40]:
collection_name = "arabic_news"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    ),
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)
print("Collection created successfully.", collection_name)

Collection created successfully. arabic_news


In [41]:
print(client.get_collection(collection_name))

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors={'sparse': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=False, datatype=None), modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=No

In [42]:
from qdrant_client.models import PointStruct , SparseVector
from fastembed import SparseTextEmbedding
from langchain_community.embeddings import HuggingFaceEmbeddings
import uuid

In [43]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1350.22it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

In [45]:
def index_documents(docs, batch_size=64):
    total=len(docs)
    for i in range(0, total, batch_size):
        batch = docs[i:i+batch_size]
        texts= [d['text']for d in batch]

        dense_vecs = embeddings.embed_documents(texts)
        sparse_vecs = list(sparse_model.embed(texts))

        points = []
        for j, doc in enumerate(batch):
            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector={
                "":dense_vecs[j],
                "sparse": SparseVector(
                    indices=sparse_vecs[j].indices.tolist(),
                    values=sparse_vecs[j].values.tolist()
                )
                },
                payload={
                    "text": doc['text'],
                    "category": doc['category'],
                    "source_idx": doc['source_idx']
                }
            ))
        client.upsert(
            collection_name=collection_name,
            points=points
        )
        if i % 640 ==0:
            print(f"Indexed {min(i+batch_size, total)}/{total} documents.")
index_documents(docs)
print('done') 

Indexed 64/30148 documents.
Indexed 704/30148 documents.
Indexed 1344/30148 documents.
Indexed 1984/30148 documents.
Indexed 2624/30148 documents.
Indexed 3264/30148 documents.
Indexed 3904/30148 documents.
Indexed 4544/30148 documents.
Indexed 5184/30148 documents.
Indexed 5824/30148 documents.
Indexed 6464/30148 documents.
Indexed 7104/30148 documents.
Indexed 7744/30148 documents.
Indexed 8384/30148 documents.
Indexed 9024/30148 documents.
Indexed 9664/30148 documents.
Indexed 10304/30148 documents.
Indexed 10944/30148 documents.
Indexed 11584/30148 documents.
Indexed 12224/30148 documents.
Indexed 12864/30148 documents.
Indexed 13504/30148 documents.
Indexed 14144/30148 documents.
Indexed 14784/30148 documents.
Indexed 15424/30148 documents.
Indexed 16064/30148 documents.
Indexed 16704/30148 documents.
Indexed 17344/30148 documents.
Indexed 17984/30148 documents.
Indexed 18624/30148 documents.
Indexed 19264/30148 documents.
Indexed 19904/30148 documents.


C:\Users\saleen\AppData\Local\Temp\ipykernel_20232\2453505764.py:27: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20032 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Indexed 20544/30148 documents.
Indexed 21184/30148 documents.
Indexed 21824/30148 documents.
Indexed 22464/30148 documents.
Indexed 23104/30148 documents.
Indexed 23744/30148 documents.
Indexed 24384/30148 documents.
Indexed 25024/30148 documents.
Indexed 25664/30148 documents.
Indexed 26304/30148 documents.
Indexed 26944/30148 documents.
Indexed 27584/30148 documents.
Indexed 28224/30148 documents.
Indexed 28864/30148 documents.
Indexed 29504/30148 documents.
Indexed 30144/30148 documents.
done


In [1]:
%pip install langchain-huggingface


  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
query = "ما هي آخر الأخبار السياسية؟"
dense_vec = embeddings.embed_query(query)
sparse_vec = list(sparse_model.embed([query]))[0]

results = client.query_points(
    collection_name="arabic_news",
    prefetch=[
        models.Prefetch(query=dense_vec, using="", limit=20),
        models.Prefetch(
            query=SparseVector(
                indices=sparse_vec.indices.tolist(),
                values=sparse_vec.values.tolist()
            ),
            using="sparse",
            limit=20
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3
)

for r in results.points:
    print(f"[{r.payload['category']}] {r.payload['text'][:120]}...")

NameError: name 'embeddings' is not defined